In [3]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd


def find_results_dir() -> Path:
    """Encontra artifacts/results subindo a árvore de diretórios a partir do cwd."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "artifacts" / "results"
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar o diretório artifacts/results a partir do diretório atual.")


def find_score_file_for_folder(folder: Path):
    """
    Procura arquivos scores*.json dentro da pasta.
    Se houver mais de um, retorna o mais recentemente modificado.
    """
    matches = sorted(folder.rglob("scores*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None


def to_float_or_none(value):
    """Converte valor para float quando possível; caso contrário, retorna None."""
    try:
        if value is None:
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def normalize_metrics(run_folder_name: str, data: dict) -> dict:
    """
    Normaliza métricas para o esquema do baseline:
    overall, relevancy, factuality, bert, rouge, similarity, bleurt, medcat, align

    Regras:
    - baseline: já vem no formato final
    - rag_*: score -> relevancy, score_secondary -> factuality,
             overall = (factuality + relevancy) / 2 se ambos existirem, senão None
    """
    is_baseline = run_folder_name == "baseline"

    if is_baseline:
        relevancy = to_float_or_none(data.get("relevancy"))
        factuality = to_float_or_none(data.get("factuality"))
        overall = to_float_or_none(data.get("overall"))
    else:
        relevancy = to_float_or_none(data.get("score"))
        factuality = to_float_or_none(data.get("score_secondary"))

        if relevancy is None or factuality is None:
            overall = None
        else:
            overall = (factuality + relevancy) / 2

    normalized = {
        "overall": overall,
        "relevancy": relevancy,
        "factuality": factuality,
        "bert": to_float_or_none(data.get("bert")),
        "rouge": to_float_or_none(data.get("rouge")),
        "similarity": to_float_or_none(data.get("similarity")),
        "bleurt": to_float_or_none(data.get("bleurt")),
        "medcat": to_float_or_none(data.get("medcat")),
        "align": to_float_or_none(data.get("align")),
    }

    return normalized


def load_score_row(run_folder: Path, score_file: Path, results_dir: Path) -> dict:
    row = {
        "run_folder": run_folder.name,
        "score_file": str(score_file.relative_to(results_dir)),
        "modified_at": datetime.fromtimestamp(score_file.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
    }

    text = score_file.read_text(encoding="utf-8").strip()
    if not text:
        row["json_error"] = "empty file"
        return row

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        row["json_error"] = f"{e.msg} (line {e.lineno}, col {e.colno})"
        row["raw_json"] = text[:500]
        return row

    if not isinstance(data, dict):
        row["raw_json"] = str(data)
        return row

    row.update(normalize_metrics(run_folder.name, data))

    # Mantém os campos originais de score quando existirem para auditoria
    if "score" in data:
        row["score"] = to_float_or_none(data.get("score"))
    if "score_secondary" in data:
        row["score_secondary"] = to_float_or_none(data.get("score_secondary"))

    return row


results_dir = find_results_dir()
run_folders = sorted([
    p for p in results_dir.iterdir()
    if p.is_dir() and p.name != "compare_results"
])

rows = []
missing = []

for folder in run_folders:
    score_file = find_score_file_for_folder(folder)
    if score_file is None:
        missing.append(folder.name)
        continue
    rows.append(load_score_row(folder, score_file, results_dir))

df = pd.DataFrame(rows)

if not df.empty:
    # Ordena pela métrica principal normalizada
    sort_cols = [c for c in ["overall", "relevancy", "factuality"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(by=sort_cols, ascending=False).reset_index(drop=True)

    # Ordem final: identificação + métricas baseline + colunas auxiliares
    baseline_metric_cols = [
        "overall", "relevancy", "factuality",
        "bert", "rouge", "similarity", "bleurt", "medcat", "align"
    ]
    lead_cols = [c for c in ["run_folder", "score_file", "modified_at"] if c in df.columns]
    metric_cols = [c for c in baseline_metric_cols if c in df.columns]
    aux_cols = [c for c in ["score", "score_secondary", "json_error", "raw_json"] if c in df.columns]

    other_cols = [c for c in df.columns if c not in lead_cols + metric_cols + aux_cols]
    df = df[lead_cols + metric_cols + aux_cols + other_cols]

print(f"Diretório analisado: {results_dir}")
print(f"Pastas com score encontrado: {len(rows)}")
print(f"Pastas sem score: {len(missing)}")
if missing:
    print("Sem scores*.json:", ", ".join(missing))

display(df)

# Opcional: salvar tabela consolidada
output_csv = results_dir / "compare_results" / "scores_comparativo.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"Tabela salva em: {output_csv}")

Diretório analisado: /home/ia368/projetos/imageclef2026-rag/artifacts/results
Pastas com score encontrado: 11
Pastas sem score: 7
Sem scores*.json: rag_20260325_195559, rag_fine_tune_full_20260404_171107, rag_fine_tune_full_20260404_172115, rag_medgemma27b_20260325_115429, rag_medgemma_27b_20260318_213004, rag_simple_prompt_20260310_113904_copy, simple_prompt_fine_tune_full_20260410_095125


,run_folder,score_file,modified_at,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align,score,score_secondary
0,baseline,baseline/scores.json,2026-03-30 08:58:25,0.342700,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900,NaN,NaN
1,rag_prompt_qwen3_VL_4B_20260423_010343,rag_prompt_qwen3_VL_4B_20260423_010343/scores....,2026-04-23 08:19:00,0.331478,0.519108,0.143848,0.600307,0.217038,0.931983,0.327104,0.152968,0.134729,0.519108,0.143848
2,simple_prompt_med15_4B_finetuned_20260422_102940,simple_prompt_med15_4B_finetuned_20260422_1029...,2026-04-22 17:11:16,0.330277,0.511993,0.148561,0.587698,0.228274,0.924946,0.307056,0.154150,0.142972,0.511993,0.148561
3,rag_base_20260223_204535,rag_base_20260223_204535/scores_20260223_20453...,2026-03-09 15:59:11,0.326805,0.499181,0.154430,0.594663,0.221224,0.874318,0.306520,0.149102,0.159757,0.499181,0.154430
4,simple_prompt_qwen3_VL_4B_20260422_192239,simple_prompt_qwen3_VL_4B_20260422_192239/scor...,2026-04-23 10:05:07,0.325250,0.486996,0.163504,0.580141,0.182586,0.878392,0.306865,0.129695,0.197314,0.486996,0.163504
5,simple_prompt_fine_tune_full_20260408_084221,simple_prompt_fine_tune_full_20260408_084221/s...,2026-04-09 19:45:12,0.324787,0.495646,0.153928,0.580538,0.219084,0.892597,0.290366,0.147047,0.160809,0.495646,0.153928
6,rag_fine_tune_full_20260406_134649,rag_fine_tune_full_20260406_134649/scores.json,2026-04-06 20:18:47,0.306847,0.482912,0.130783,0.575279,0.196777,0.867951,0.291639,0.139918,0.121648,0.482912,0.130783
7,rag_few_shot_20260312_103549,rag_few_shot_20260312_103549/scores.json,2026-03-22 18:40:49,0.304593,0.478106,0.131079,0.581801,0.205245,0.835839,0.289540,0.112642,0.149517,0.478106,0.131079
8,few_shot_fine_tuning_full_20260409_093749,few_shot_fine_tuning_full_20260409_093749/scor...,2026-04-09 19:50:42,0.304068,0.478473,0.129664,0.582002,0.205553,0.835759,0.290578,0.112550,0.146777,0.478473,0.129664
9,rag_simple_prompt_20260310_113904,rag_simple_prompt_20260310_113904/scores.json,2026-03-22 18:51:23,0.303262,0.457017,0.149508,0.568059,0.183479,0.813368,0.263162,0.102806,0.196209,0.457017,0.149508


Tabela salva em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/compare_results/scores_comparativo.csv


In [4]:
df.drop(columns=["score_file", "score", "score_secondary"])

,run_folder,modified_at,overall,relevancy,factuality,bert,rouge,similarity,bleurt,medcat,align
0,baseline,2026-03-30 08:58:25,0.342700,0.534400,0.151100,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900
1,rag_prompt_qwen3_VL_4B_20260423_010343,2026-04-23 08:19:00,0.331478,0.519108,0.143848,0.600307,0.217038,0.931983,0.327104,0.152968,0.134729
2,simple_prompt_med15_4B_finetuned_20260422_102940,2026-04-22 17:11:16,0.330277,0.511993,0.148561,0.587698,0.228274,0.924946,0.307056,0.154150,0.142972
3,rag_base_20260223_204535,2026-03-09 15:59:11,0.326805,0.499181,0.154430,0.594663,0.221224,0.874318,0.306520,0.149102,0.159757
4,simple_prompt_qwen3_VL_4B_20260422_192239,2026-04-23 10:05:07,0.325250,0.486996,0.163504,0.580141,0.182586,0.878392,0.306865,0.129695,0.197314
5,simple_prompt_fine_tune_full_20260408_084221,2026-04-09 19:45:12,0.324787,0.495646,0.153928,0.580538,0.219084,0.892597,0.290366,0.147047,0.160809
6,rag_fine_tune_full_20260406_134649,2026-04-06 20:18:47,0.306847,0.482912,0.130783,0.575279,0.196777,0.867951,0.291639,0.139918,0.121648
7,rag_few_shot_20260312_103549,2026-03-22 18:40:49,0.304593,0.478106,0.131079,0.581801,0.205245,0.835839,0.289540,0.112642,0.149517
8,few_shot_fine_tuning_full_20260409_093749,2026-04-09 19:50:42,0.304068,0.478473,0.129664,0.582002,0.205553,0.835759,0.290578,0.112550,0.146777
9,rag_simple_prompt_20260310_113904,2026-03-22 18:51:23,0.303262,0.457017,0.149508,0.568059,0.183479,0.813368,0.263162,0.102806,0.196209
